In [1]:
import torch
import numpy as np
import os
import glob
from accelerate import Accelerator
from transformers import Sam3VideoModel, Sam3VideoProcessor
from transformers import Sam3TrackerVideoModel, Sam3TrackerVideoProcessor
from PIL import Image

# Step 1
Get the masks via text prompt

In [2]:
device = Accelerator().device
model = Sam3VideoModel.from_pretrained("facebook/sam3").to(device, dtype=torch.bfloat16)
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

In [3]:
video_path = "./videos/frames"
video_frames_paths = glob.glob(os.path.join(video_path, "*.jpg"))
video_frames_paths.sort(key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
video_frames = [np.array(Image.open(f)) for f in video_frames_paths]

In [16]:
inference_session = processor.init_video_session(
    video=video_frames,
    inference_device=device,
    processing_device="cpu",
    video_storage_device="cpu",
    dtype=torch.bfloat16,
)
# Add text prompt to detect and track objects
text = "fish"
inference_session = processor.add_text_prompt(
    inference_session=inference_session,
    text=text,
)
outputs_per_frame = {}
for model_outputs in model.propagate_in_video_iterator(
    inference_session=inference_session, max_frame_num_to_track=50
):
    processed_outputs = processor.postprocess_outputs(inference_session, model_outputs)
    outputs_per_frame[model_outputs.frame_idx] = processed_outputs

print(f"Processed {len(outputs_per_frame)} frames")

Processed 5 frames


In [17]:

outputs_per_frame = {}
for model_outputs in model.propagate_in_video_iterator(
    inference_session=inference_session, max_frame_num_to_track=50
):
    processed_outputs = processor.postprocess_outputs(inference_session, model_outputs)
    outputs_per_frame[model_outputs.frame_idx] = processed_outputs

print(f"Processed {len(outputs_per_frame)} frames")

Processed 5 frames


# Step 2
Use Sam3TrackerVideo for refinements

### First, we load the model

In [18]:
trackerModel = Sam3TrackerVideoModel.from_pretrained("facebook/sam3").to(device, dtype=torch.bfloat16)
trackerProcessor = Sam3TrackerVideoProcessor.from_pretrained("facebook/sam3")

Loading weights:   0%|          | 0/845 [00:00<?, ?it/s]

In [19]:
inference_session = trackerProcessor.init_video_session(
    video=video_frames,
    inference_device=device,
    dtype=torch.bfloat16,
)

### Then we initialize by loadling data from the first frame. After initialization we load the other frames
based on https://huggingface.co/docs/transformers/model_doc/sam3_tracker_video#transformers.Sam3TrackerVideoModel

In [20]:
# 1. Schritt: Masken für den START-Frame (0) hinzufügen
frame_idx = 0
trackerProcessor.add_inputs_to_inference_session(
    inference_session=inference_session,
    frame_idx=frame_idx,
    obj_ids=outputs_per_frame[frame_idx]['object_ids'].tolist(),
    input_masks=list(outputs_per_frame[frame_idx]['masks']),
)

# 2. Schritt: Den Tracker auf Frame 0 ausführen mit run_mem_encoder=True
# Das ist der entscheidende Punkt: Hier wird die Memory Bank gefüllt!
trackerModel(
    inference_session=inference_session,
    frame_idx=frame_idx,
    run_mem_encoder=True  # Zwingt das Modell, die Masken in den Speicher zu schreiben
)

# 3. Schritt: Jetzt kannst du mit dem Tracking für den Rest fortfahren
# (Hier kannst du entweder propagieren oder weitere Masken hinzufügen)
for frame in range(1, len(video_frames)):
    # Wenn du für diesen Frame bereits eine Maske hast (vom SAM3 Modell):
    trackerProcessor.add_inputs_to_inference_session(
        inference_session=inference_session,
        frame_idx=frame,
        obj_ids=outputs_per_frame[frame]['object_ids'].tolist(),
        input_masks=list(outputs_per_frame[frame]['masks']),
    )
    
    # Inferenz für diesen Frame (er nutzt jetzt das Gedächtnis von Frame 0)
    outputs = trackerModel(inference_session=inference_session, frame_idx=frame)

In [21]:
video_res_masks = trackerProcessor.post_process_masks(
    [outputs.pred_masks], original_sizes=[[inference_session.video_height, inference_session.video_width]], binarize=False
)[0]
print(f"Generated masks for {video_res_masks.shape[0]} objects")

Generated masks for 22 objects


In [22]:
video_segments = {}
for sam3_tracker_video_output in trackerModel.propagate_in_video_iterator(inference_session):
    video_res_masks = trackerProcessor.post_process_masks(
        [sam3_tracker_video_output.pred_masks], original_sizes=[[inference_session.video_height, inference_session.video_width]], binarize=False
    )[0]
    video_segments[sam3_tracker_video_output.frame_idx] = {
        obj_id: video_res_masks[i]
        for i, obj_id in enumerate(inference_session.obj_ids)
    }

print(f"Tracked {len(inference_session.obj_ids)} objects through {len(video_segments)} frames")

Tracked 22 objects through 5 frames


### Next, we want to visualize our results

In [23]:
from sam3.visualization_utils import (
    load_frame,
    prepare_masks_for_visualization,
    visualize_formatted_frame_output,
)

We need to convert our outputs appropriately we can use the given functions

In [24]:
import torch

outputs_new_format = []

for frame in range(len(video_frames)):
    frame_data = {}
    
    # 1. IDs des AKTUELLEN Frames holen (nicht nur von Frame 0)
    # Wir wandeln sie in eine Liste und dann in einen Tensor um
    current_ids = list(video_segments[frame].keys())
    frame_data['out_obj_ids'] = torch.tensor([int(i) for i in current_ids]).cpu()
    
    # 2. Masken sammeln
    masks_list = []
    for obj_id in current_ids:
        # Die Maske holen und direkt auf die CPU schieben
        mask = video_segments[frame][obj_id][0].cpu()
        
        # WICHTIG: Hier binarisieren! 
        # SAM Logits sind > 0 für das Objekt.
        binary_mask = (mask > 0.0) 
        
        masks_list.append(binary_mask)
    
    # 3. Masken zu einem Tensor der Form [Anzahl_Objekte, H, W] zusammenfügen
    if masks_list:
        frame_data['out_binary_masks'] = torch.stack(masks_list)
    else:
        # Falls in einem Frame mal kein Fisch gefunden wurde
        frame_data['out_binary_masks'] = torch.empty(0)
    
    # Optional: Falls du Boxen hast, kannst du sie hier auch hinzufügen:
    # frame_data['out_boxes_xywh'] = ... 

    outputs_new_format.append(frame_data)

print(f"Daten für {len(outputs_new_format)} Frames erfolgreich umgeformt.")

Daten für 5 Frames erfolgreich umgeformt.


In [25]:
import os
import matplotlib
matplotlib.use('Agg') # Hintergrund-Modus
import matplotlib.pyplot as plt
from PIL import Image
import io

# Deaktiviert plt.show() Warnungen
plt.show = lambda: None 

output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)

for frame_idx in range(len(outputs_new_format)):
    plt.close('all')
    
    # Visualisierung aufrufen
    visualize_formatted_frame_output(
        frame_idx,
        video_frames,
        outputs_list=[prepare_masks_for_visualization({frame_idx: outputs_new_format[frame_idx]})],
        titles=["SAM 3 Tracking"],
        figsize=(10, 7),
    )
    
    # Bild in Puffer speichern
    buf = io.BytesIO()
    plt.savefig(buf, format='jpg', bbox_inches='tight', pad_inches=0.05, dpi=150)
    buf.seek(0)
    
    # Bild laden, drehen und speichern
    img = Image.open(buf)
    # ROTATE_90 = 90 Grad gegen Uhrzeigersinn
    img_rotated = img.transpose(Image.ROTATE_270)
    
    file_name = f"output{frame_idx + 1:04d}.jpg"
    img_rotated.save(os.path.join(output_dir, file_name), quality=95)
    
    buf.close()
    
    if (frame_idx + 1) % 5 == 0:
        print(f"Frame {frame_idx + 1} verarbeitet.")

print(f"Fertig! Alle Bilder in {output_dir}")

Frame 5 verarbeitet.
Fertig! Alle Bilder in ./outputs
